<a href="https://colab.research.google.com/github/springboardmentor123g/PlantDocBotProject/blob/Intern-UjjwalKumar/Text_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [8]:
import pandas as pd
from datasets import load_dataset

In [9]:
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")

In [12]:
df.drop(columns=["image"], inplace=True)

In [13]:
df

,caption,captions
0,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."
...,...,...
20633,Potato Early blight,[A potato leaf with concentric brown rings for...
20634,Tomato Spider mites Two spotted spider mite,[A tomato leaf infested with two-spotted spide...
20635,Potato Late blight,"[A potato leaf showing dark, water-soaked lesi..."
20636,Tomato Spider mites Two spotted spider mite,[A tomato leaf infested with two-spotted spide...


In [14]:
df.to_csv("Textdata.csv", index=False)

In [15]:
df = pd.read_csv("Textdata.csv")

In [18]:
df.explode("captions")


,caption,captions
0,Tomato healthy,['A vibrant green and healthy tomato leaf with...
1,Tomato Late blight,['A tomato leaf showing dark brown lesions and...
2,Tomato healthy,['A vibrant green and healthy tomato leaf with...
3,Tomato mosaic virus,['A tomato leaf with mosaic-like patterns of l...
4,Pepper bell healthy,['A fresh green bell pepper leaf with a smooth...
...,...,...
20633,Potato Early blight,['A potato leaf with concentric brown rings fo...
20634,Tomato Spider mites Two spotted spider mite,['A tomato leaf infested with two-spotted spid...
20635,Potato Late blight,"['A potato leaf showing dark, water-soaked les..."
20636,Tomato Spider mites Two spotted spider mite,['A tomato leaf infested with two-spotted spid...


In [19]:
from sklearn.preprocessing import LabelEncoder

In [21]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["caption"])

In [53]:
df[["caption", "label"]].head()


,caption,label
0,Tomato healthy,13
1,Tomato Late blight,7
2,Tomato healthy,13
3,Tomato mosaic virus,14
4,Pepper bell healthy,1


In [54]:
num_classes = len(label_encoder.classes_)
print("Number of classes:", num_classes)

Number of classes: 15


In [58]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    train_size=0.8,
    random_state=42
)


In [24]:
from transformers import AutoTokenizer

In [25]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [37]:
def tokenize_function(data):
    return tokenizer(
        data["captions"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


In [60]:
from datasets import Dataset

# Convert pandas DataFrames to Dataset objects
train_hf_dataset = Dataset.from_pandas(train_df)
test_hf_dataset = Dataset.from_pandas(test_df) # Use test_df as the validation dataset source

# Tokenize the datasets
train_tokenized_dataset = train_hf_dataset.map(tokenize_function, batched=True)
test_tokenized_dataset = test_hf_dataset.map(tokenize_function, batched=True)

# Prepare datasets for the Trainer
# The Trainer expects 'labels' column, so rename 'label' to 'labels'
# Also remove pandas specific index column if present
train_dataset = train_tokenized_dataset.rename_column("label", "labels").remove_columns(["caption", "captions", "__index_level_0__"])
val_ds = test_tokenized_dataset.rename_column("label", "labels").remove_columns(["caption", "captions", "__index_level_0__"])

Map:   0%|          | 0/16510 [00:00<?, ? examples/s]

Map:   0%|          | 0/4128 [00:00<?, ? examples/s]

In [28]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [29]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=9
)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

In [31]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)



In [63]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

training_args = TrainingArguments(
    output_dir="./bert_symptom_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    logging_steps=100,
    report_to="none",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-684369736.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.000100,0.000036,1.000000
2,0.000000,0.000009,1.000000


TrainOutput(global_step=8256, training_loss=0.019977192560485166, metrics={'train_runtime': 357.9474, 'train_samples_per_second': 92.248, 'train_steps_per_second': 23.065, 'total_flos': 1093771893427200.0, 'train_loss': 0.019977192560485166, 'epoch': 2.0})

In [64]:
# save model and tokenizer
model.save_pretrained("./plant_disease_model")
tokenizer.save_pretrained("./plant_disease_model")


('./plant_disease_model/tokenizer_config.json',
 './plant_disease_model/special_tokens_map.json',
 './plant_disease_model/vocab.txt',
 './plant_disease_model/added_tokens.json',
 './plant_disease_model/tokenizer.json')

In [65]:
import torch
from transformers import pipeline

# create inference pipeline
inference_pipeline = pipeline(
    "text-classification",
    model="./plant_disease_model",
    tokenizer="./plant_disease_model",
    device=0 if torch.cuda.is_available() else -1
)

# example prediction
result = inference_pipeline("A tomato leaf showing dark brown lesions")
print(result)


Device set to use cuda:0


[{'label': 'LABEL_3', 'score': 0.9728665351867676}]


In [66]:
import joblib

joblib.dump(label_encoder, "label_encoder.pkl")


['label_encoder.pkl']

In [67]:
label_encoder = joblib.load("label_encoder.pkl")


In [69]:
pred_label = result[0]["label"]
class_id = int(pred_label.replace("LABEL_", ""))
disease_name = label_encoder.inverse_transform([class_id])[0]
print(disease_name)


Potato Late blight


In [70]:
import shutil

shutil.make_archive(
    "plant_disease_model",
    "zip",
    "./plant_disease_model"
)


'/content/plant_disease_model.zip'

In [71]:
texts = [
    "A potato leaf showing dark, water-soaked lesions and a white moldy growth, indicative of late blight.",
    "A potato leaf with large irregular brown patches spreading rapidly across the surface."
]

results = inference_pipeline(texts)

for text, res in zip(texts, results):
    class_id = int(res["label"].replace("LABEL_", ""))
    disease_name = label_encoder.inverse_transform([class_id])[0]
    print(f"Text: {text}")
    print(f"Prediction: {disease_name} | Confidence: {res['score']:.4f}")
    print("-" * 60)


Text: A potato leaf showing dark, water-soaked lesions and a white moldy growth, indicative of late blight.
Prediction: Potato Late blight | Confidence: 1.0000
------------------------------------------------------------
Text: A potato leaf with large irregular brown patches spreading rapidly across the surface.
Prediction: Tomato Target Spot | Confidence: 0.6203
------------------------------------------------------------
